In [1]:
!pip install fastapi uvicorn transformers accelerate sentencepiece bitsandbytes


In [ ]:
import requests
from dotenv import load_dotenv
import os
import spacy
import xml.etree.ElementTree as ET
import json
import requests
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
import os
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
load_dotenv()

True

In [ ]:
import os
import requests
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue

LABEL_STUDIO_URL = "http://localhost:8081"
LABEL_STUDIO_API_KEY = os.getenv("LABEL_STUDIO")
PROJECT_ID = 1  # replace with your project ID
MODEL_URL = "https://empiristic-mariyah-unprophetically.ngrok-free.dev/predict"

client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY)
project = client.projects.get(id=PROJECT_ID)

# Get the parsed labeling interface to build valid prediction payloads
li = project.get_label_interface()

# Iterate tasks and attach predictions
for task in client.tasks.list(project=project.id):
    task_id = task.id
    data = task.data or {}

    text = data.get("text", "")
    print(text)

    # Call your model
    resp = requests.post(
        MODEL_URL,
        json={"data": [{"text": text}]},
        timeout=300,
    )
    resp.raise_for_status()
    result = resp.json()

    # Normalize: accept either a list of spans OR a dict with "result"
    spans = result.get("result", result) if isinstance(result, dict) else result

    # Build ONE PredictionValue per task (Label Studio expects result=[...])
    prediction = PredictionValue(
        model_version="mistralner",
        score=float(result.get("score", 0.99)) if isinstance(result, dict) else 0.99,
        result=spans,
    )

    # Create prediction in Label Studio
    client.predictions.create(task=task_id, **prediction.model_dump())


In [ ]:
##testing

import requests

MODEL_URL = "https://empiristic-mariyah-unprophetically.ngrok-free.dev/predict"

payload = {
    "data": [
        {"text": "A temperature sensor is connected to a control unit via a wireless interface."}
    ]
}

resp = requests.post(MODEL_URL, json=payload, timeout=60)
resp.raise_for_status()

print("Status:", resp.status_code)
print("Response JSON:")
print(resp.json())


In [ ]:

# ---------------- CONFIG ----------------
LABEL_STUDIO_URL = "http://localhost:8081"   # EXACT URL you open LS in browser
LABEL_STUDIO_API_KEY = os.getenv("LABEL_STUDIO")
PROJECT_ID = 2

MODEL_URL = "https://empiristic-mariyah-unprophetically.ngrok-free.dev/predict"
MODEL_VERSION = "mistral8B"

# Control names from your Label Studio labeling config
FROM_NAME = "ner"     # e.g. <Labels name="ner" ...>
TO_NAME = "text"      # e.g. <Text name="text" ...>

client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY.strip())
project = client.projects.get(id=PROJECT_ID)

# optional: keep this if you want parity with docs (not required anymore)
li = project.get_label_interface()

print(f"Connected. Project={project.id}")

for task in client.tasks.list(project=project.id):
    task_id = task.id
    data = task.data or {}

    text = data.get("text", "")
    if not isinstance(text, str) or not text.strip():
        print(f"Task {task_id}: no 'text' field, skipping.")
        continue

    # Call your Colab FastAPI /predict (expects {"data":[{"text":...}]})
    resp = requests.post(
        MODEL_URL,
        json={"data": [{"text": text}]},
        timeout=300,
    )

    # If something is wrong, this prints the FastAPI validation error
    if resp.status_code != 200:
        print(f"Task {task_id}: model error {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

    model_preds = resp.json()
    if isinstance(model_preds, dict):
        model_preds = [model_preds]

    if not model_preds:
        print(f"Task {task_id}: model returned empty list.")
        continue

    # Your /predict returns a list of prediction dicts like:
    # {"result": [...], "score": 1.0, "model_version": "..."}
    # We'll store each one as an LS Prediction.
    saved_any = False
    for mp in model_preds:
        result = mp.get("result")
        if not isinstance(result, list) or len(result) == 0:
            continue

        prediction = PredictionValue(
            model_version=mp.get("model_version", MODEL_VERSION),
            score=float(mp.get("score", 0.0)),
            result=result,  # <-- pass through as-is
        )

        client.predictions.create(task=task_id, **prediction.model_dump())
        saved_any = True

    if saved_any:
        print(f"✔ Saved prediction(s) for task {task_id}")
    else:
        # Debug: show the first model prediction keys to see what's missing
        print(f"Task {task_id}: no usable 'result'. First pred keys: {list(model_preds[0].keys())}")


Connected. Project=2
Task 14021: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14022: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14023: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14024: no usable 'result'. First pred keys: ['result', 'score', 'model_version']
Task 14025: no usable 'result'. First pred keys: ['result', 'score', 'model_version']


KeyboardInterrupt: 